# FER2013: Facial Expression Recognition

სახის ემოციების კლასიფიკაცია PyTorch-ის გამოყენებით. 3 არქიტექტურა, 2 hyperparameter sweep, wandb ლოგირება.

## 1. გარემოს მომზადება

რეპოზიტორიის კლონირება, ბიბლიოთეკების დაყენება, credentials, მონაცემები.

In [ ]:
import os, getpass

# რეპოზიტორია
!git clone https://github.com/GigiSichinava/ML-Assignment-4.git /content/ML-Assignment-4 2>/dev/null || echo 'already cloned'
os.chdir('/content/ML-Assignment-4')
!pip install -q wandb kaggle

# credentials
os.environ['KAGGLE_USERNAME'] = input('kaggle username: ').strip()
os.environ['KAGGLE_KEY']      = getpass.getpass('kaggle key: ').strip()
os.environ['WANDB_API_KEY']   = getpass.getpass('wandb key: ').strip()

# მონაცემები
import pandas as pd
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    !kaggle datasets download -d deadskull7/fer2013 -p data/
    !unzip -q -o data/fer2013.zip -d data/
    df = pd.read_csv('data/fer2013.csv')
    df[df['Usage']=='Training'][['emotion','pixels']].to_csv('data/train.csv', index=False)
    df[df['Usage']!='Training'][['pixels']].to_csv('data/test.csv', index=False)
    print('train.csv და test.csv შეიქმნა')
else:
    print('მონაცემები უკვე ადგილზეა')

# დაკავშირება wandb-თან
import wandb
wandb.login()

# იმპორტები
import torch
from src.config import Config, PRESETS
from src.data import get_dataloaders
from src.models import get_model
from src.train import train_model, make_submission
from src.utils import set_seed, count_params, check_initial_loss, overfit_small_batch, plot_history, plot_confusion

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(42)
print('ready, device:', device)

## 2. Sanity Check-ები (Forward და Backward)

ლექციაზე განხილული შემოწმებები სრული ტრენინგის დაწყებამდე.

**Forward:** random მოდელის loss ~ ln(7) = 1.946. თუ სცდება, loss/softmax-ში ბაგია.

**Backward:** 20 მაგალითზე ~100% train acc. თუ ვერ აღწევს, training loop-ში ბაგია.

In [ ]:
cfg = Config(arch='SmallCNN')
train_loader, val_loader, test_loader, class_weights = get_dataloaders(cfg)
model = get_model(cfg).to(device)
print('params:', count_params(model))

# forward შემოწმება
check_initial_loss(model, train_loader, device)

# backward შემოწმება
overfit_small_batch(get_model(cfg).to(device), train_loader, device, n=20, steps=200)

## 3. ექსპერიმენტი 1: BaselineMLP

სურათს ვაბრტყელებ ვექტორად და 2 FC layer-ს ვიყენებ. spatial სტრუქტურა იკარგება. ველოდებით overfit-ს: მოდელი train set-ს ზეპირად ისწავლის, val-ზე ვერ განაზოგადებს.

In [ ]:
cfg = PRESETS['mlp_baseline']
model_mlp, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title='BaselineMLP')
print('best val acc:', round(best, 3))

## 4. ექსპერიმენტი 2: SmallCNN

2 conv block + BatchNorm. 48x48 -> 24x24 -> 12x12. Convolution spatial feature-ებს ინარჩუნებს, ამიტომ MLP-ზე მკვეთრ გაუმჯობესებას ველით.

In [ ]:
cfg = PRESETS['smallcnn']
model_small, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title='SmallCNN')
plot_confusion(y_true, y_pred)
print('best val acc:', round(best, 3))

## 5. ექსპერიმენტი 3: DeeperCNN (Regularization-ის გარეშე)

4 conv block, dropout=0, გამორთული augmentation. განზრახ ვტოვებ regularization-ის გარეშე, რომ overfit-ი ნათლად ჩანდეს. ველოდებით: train acc ~100%, val acc-ის გაჩერებას, val loss-ის ზრდას.

In [ ]:
cfg = PRESETS['deepcnn_overfit']
model_deep_ov, history, best, _ = train_model(cfg, device=device)
plot_history(history, title='DeeperCNN (no regularization)')
print('best val acc:', round(best, 3))

## 6. ექსპერიმენტი 4: DeeperCNN (Regularized)

იგივე 4 conv block, მაგრამ dropout=0.4, weight_decay=1e-4, augmentation და cosine LR scheduler. ველოდებით overfit_gap-ის შემცირებას და val acc-ის ზრდას.

In [ ]:
cfg = PRESETS['deepcnn_regularized']
model_best, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title='DeeperCNN (regularized)')
plot_confusion(y_true, y_pred)
print('best val acc:', round(best, 3))

## 7. Hyperparameter Sweep: SmallCNN, სხვადასხვა Learning Rate

lr-ის გავლენა სწავლებაზე. ველოდებით: lr=0.01 არასტაბილური, lr=0.001 ოპტიმალური, lr=0.0001 ნელი კონვერგენცია.

In [ ]:
for lr in [1e-2, 1e-3, 1e-4]:
    cfg = Config(arch='SmallCNN', run_name=f'smallcnn_lr{lr}',
                 lr=lr, dropout=0.25, epochs=25)
    _, history, best, _ = train_model(cfg, device=device)
    print(f'lr={lr}: best val acc = {round(best, 3)}')

## 8. Hyperparameter Sweep: DeeperCNN, სხვადასხვა Dropout

dropout-ის გავლენა overfit_gap-ზე. ველოდებით: dropout=0.0 მაღალი acc მაგრამ დიდი gap, dropout=0.5 მცირე gap მაგრამ დაბალი acc.

In [ ]:
for d in [0.0, 0.3, 0.5]:
    cfg = Config(arch='DeeperCNN', run_name=f'deepcnn_drop{d}',
                 dropout=d, weight_decay=1e-4, augment=True,
                 scheduler='cosine', epochs=40)
    _, history, best, _ = train_model(cfg, device=device)
    print(f'dropout={d}: best val acc = {round(best, 3)}')

## 9. Kaggle Submission

საუკეთესო მოდელით (DeeperCNN regularized) test set-ზე prediction.

In [ ]:
make_submission(model_best, PRESETS['deepcnn_regularized'],
                out_path='submission.csv', device=device)